In [33]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
import os
import tqdm

domain_id_name_map = {
    "digitsdg": {
        0: 'MNIST', 1: 'MNIST-M',  2: 'SVHN', 3: 'SYN'
    },
    "officehome": {
        0: 'Art', 1: 'Clipart', 2: 'Product', 3: 'Real World'
    }
}

data_folder = "../imgs/digitsdg_dot00"
# data_folder = "../imgs/officehome_dot00"
data_path = os.path.join(data_folder, "features.npz")

img_output_folder = os.path.join(data_folder, "tsne_imgs")
if not os.path.exists(img_output_folder):
    os.makedirs(img_output_folder)

data = np.load(data_path)
all_features = data["features"]  # [num_samples, 13, 768]
domain_ids = data["domain_ids"]  # 域标签
class_ids = data["class_ids"]    # 类标签

dataset = data_folder.split("/")[2].split("_")[0]
print(f"Dataset: {dataset}")
print(f"Number of samples: {all_features.shape[0]}")
print(f"Number of layers: {all_features.shape[1]}")
print(f"Number of features: {all_features.shape[2]}")

print(f"Number of domains: {len(np.unique(domain_ids))}")
print(f"Number of classes: {len(np.unique(class_ids))}")

TSNE_PARAMS = {
    "n_components": 2,
    "perplexity": 30,         # 根据数据密度调整（5-50
    "max_iter": 1000,           # 确保收敛
    "learning_rate": 200,     # 默认值200，高维数据可增大
    "random_state": 42,       # 固定随机性
    "angle": 0.3              # Barnes-Hut加速参数
}
DOMAIN_COLOR_PALETTE = sns.color_palette("deep", n_colors=len(np.unique(domain_ids)))  # 高区分度配色
CLASS_COLOR_PALETTE = sns.color_palette("deep", n_colors=len(np.unique(class_ids)))  # 低区分度配色

for layer_idx in tqdm.trange(13):
    features_layer = all_features[:, layer_idx, :]
    features_normalized = (features_layer - np.mean(features_layer, axis=0)) / np.std(features_layer, axis=0)

    tsne = TSNE(**TSNE_PARAMS)
    embeddings_path = os.path.join(data_folder, f"tsne_layer{layer_idx}_embeddings.npy")
    if os.path.exists(embeddings_path):
        embeddings = np.load(embeddings_path)
    else:
        embeddings = tsne.fit_transform(features_normalized)
        np.save(os.path.join(data_folder, f"tsne_layer{layer_idx}_embeddings.npy"), embeddings)
    
    # 按domain_id染色绘图
    plt.figure(figsize=(4,4))
    sns.scatterplot(x=embeddings[:,0], y=embeddings[:,1], 
                    hue=domain_ids, palette=DOMAIN_COLOR_PALETTE[:len(np.unique(domain_ids))],
                    s=10, edgecolor="none", alpha=0.8, legend=(layer_idx==12))
    plt.axis("off")
    if layer_idx == 12:
        plt.legend(handles=[
            plt.Line2D([], [], marker='o', color=DOMAIN_COLOR_PALETTE[i], label=domain_id_name_map[dataset][i])
            for i in range(len(np.unique(domain_ids)))
        ], loc='upper right')
    # plt.savefig(f"{img_output_folder}/{dataset}_tsne_layer{layer_idx}_domain.pdf", dpi=600, bbox_inches="tight")
    plt.savefig(f"{img_output_folder}/{dataset}_tsne_layer{layer_idx}_domain.png", dpi=600, bbox_inches="tight", format="png")
    plt.close()
    
    # 按class_id染色绘图
    plt.figure(figsize=(4,4))
    sns.scatterplot(x=embeddings[:,0], y=embeddings[:,1], 
                    hue=class_ids, palette=CLASS_COLOR_PALETTE[:len(np.unique(class_ids))],
                    s=10, edgecolor="none", alpha=0.8, legend=False)
    plt.axis("off")
    # plt.savefig(f"{img_output_folder}/{dataset}_tsne_layer{layer_idx}_class.pdf", dpi=600, bbox_inches="tight", format="pdf")
    plt.savefig(f"{img_output_folder}/{dataset}_tsne_layer{layer_idx}_class.png", dpi=600, bbox_inches="tight", format="png")
    plt.close()

Dataset: digitsdg
Number of samples: 4800
Number of layers: 13
Number of features: 768
Number of domains: 4
Number of classes: 10


100%|██████████| 13/13 [00:15<00:00,  1.19s/it]


In [31]:
# data_folder = "../imgs/digitsdg_dot00"
data_folder = "../imgs/officehome_dot00"
for root, dirs, files in os.walk(data_folder):
    for file in files:
        if file.endswith(".pdf"):
            # remove it
            os.remove(os.path.join(root, file))